# BrainDecode: Analysis Dependencies Generation Pipeline

Generates all required dependency files per experiment from MaxQuant output and
pipeline-derived dictionaries.

Outputs (saved to `DEPDIR/{experiment}/`):

1. `dataset_metrics.xlsx`
2. `Position_probability_fragment_ion_data.csv`
3. `Fragment_ion_dict.p`
4. `SAAP_precursor_reporter_quant.xlsx`
5. `MTP_sequences.fasta`
6. `Modified_peptide_filter_dict_DP2valSAAP.p`
7. `PTM_heatmap_dict.p`
8. `PTM_heatmap_data.xlsx`
9. `fragments_per_saap_4barplot_allDS.xlsx`

Last Updated 05-18-2026 by Alex Maropakis


## Imports

In [1]:
print("Reading in packages...")

import os
import pickle
from collections import Counter
from typing import Any
import numpy as np
import pandas as pd

print("Packages loaded successfully!")


Reading in packages...
Packages loaded successfully!


## Directories

In [2]:
print("Loading directories...")

CODE_DIR        = '/Users/alexmaropakis/Projects/BrainDecode/'
PROJECT_DIR     = '/Users/alexmaropakis/Projects/Project_BrainDecode/'
INDIR           = PROJECT_DIR + 'Analysis_Inputs/'
OUTDIR          = CODE_DIR + 'Dependencies/Analysis_Outputs/'
DEPDIR          = CODE_DIR + 'Dependencies/'
MQ_DIR          = PROJECT_DIR + 'mq_output/'
PLOT_DIR        = PROJECT_DIR + 'Plots/'
SAMPLE_MAPS_DIR = DEPDIR + 'Sample_maps/'

os.makedirs(DEPDIR, exist_ok=True)
os.makedirs(OUTDIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

# ── Ping 2018 ────────────────────────────────────────────────────────────────
ping2018        = INDIR + 'Ping_2018/'
acg             = ping2018 + 'acg/'
acg_mq          = MQ_DIR + 'Ping_2018/acg/'
acg_batches     = ['b1','b2','b3','b4','b5']
acg_dirs, acg_samples, acg_maps, acg_mq_dirs = [], [], [], []
for b in acg_batches:
    _map = pd.read_excel(SAMPLE_MAPS_DIR + f'sample_map_acg{b}.xlsx')
    acg_dirs.append(acg + f'{b}/')
    acg_maps.append(_map)
    acg_samples.append(['S' + str(i) for i in sorted(set[Any](_map['TMT plex']))])
    acg_mq_dirs.append(acg_mq + f'{b}/')

fc             = ping2018 + 'fc/'
fc_mq          = MQ_DIR + 'Ping_2018/fc/'
fc_batches     = ['b1','b2','b3','b4','b5']
fc_dirs, fc_samples, fc_maps, fc_mq_dirs = [], [], [], []
for b in fc_batches:
    _map = pd.read_excel(SAMPLE_MAPS_DIR + f'sample_map_fc{b}.xlsx')
    fc_dirs.append(fc + f'{b}/')
    fc_maps.append(_map)
    fc_samples.append(['S' + str(i) for i in sorted(set[Any](_map['TMT plex']))])
    fc_mq_dirs.append(fc_mq + f'{b}/')

# ── Takasugi 2024 ────────────────────────────────────────────────────────────
ms              = INDIR + 'Takasugi_2024/'
ms_mq           = MQ_DIR + 'Takasugi_2024/'
ms_tissues      = ['Aorta', 'Brain', 'Heart', 'Kidney', 'Liver', 'Muscle', 'Skin']
ms_dirs, ms_samples, ms_maps, ms_mq_dirs = [], [], [], []
for tissue in ms_tissues:
    _aas_dir = ms + f'{tissue}/'
    _map     = pd.read_excel(SAMPLE_MAPS_DIR + f'sample_map_{tissue.lower()}.xlsx')
    ms_dirs.append(_aas_dir)
    ms_maps.append(_map)
    ms_samples.append(['S' + str(i) for i in sorted(set[Any](_map['TMT plex']))])
    ms_mq_dirs.append(ms_mq + f'{tissue}/')

# # ── Bai 2020 ─────────────────────────────────────────────────────────────────
# pooled          = INDIR + 'Bai_2020/'
# pooled_mq       = MQ_DIR + 'Bai_2020/'
# pooled_map      = pd.read_excel(SAMPLE_MAPS_DIR + 'sample_map_pooled.xlsx')
# pooled_samples  = ['S' + str(i) for i in sorted(set[Any](pooled_map['TMT plex']))]

# ── Master lists (parallel — one entry per dataset) ──────────────────────────
datasets = (
    [f'acg{b}' for b in acg_batches],
    [f'fc{b}'  for b in fc_batches],
    # ['pooled'],
    ms_tissues,
)
all_datasets    = [d for grp in datasets for d in grp]

# proj_dir_list   = acg_dirs    + fc_dirs    + [pooled]    + ms_dirs
# mq_dir_list     = acg_mq_dirs + fc_mq_dirs + [pooled_mq] + ms_mq_dirs
# samples_list    = acg_samples + fc_samples + [pooled_samples] + ms_samples
# sample_map_list = acg_maps    + fc_maps    + [pooled_map]     + ms_maps

proj_dir_list   = acg_dirs    + fc_dirs    +  ms_dirs
mq_dir_list     = acg_mq_dirs + fc_mq_dirs +  ms_mq_dirs
samples_list    = acg_samples + fc_samples +  ms_samples
sample_map_list = acg_maps    + fc_maps    +  ms_maps

# ── TMT-set to dataset name mapping ───────────────────────────────────────────
# Key is matched as a substring of proj_dir.lower().
tmt_to_dataset = {
    '/acg/b1/': {'S1': 'acgb1', 'S2': 'acgb2', 'S3': 'acgb3', 'S4': 'acgb4', 'S5': 'acgb5'},
    '/acg/b2/': {'S1': 'acgb1', 'S2': 'acgb2', 'S3': 'acgb3', 'S4': 'acgb4', 'S5': 'acgb5'},
    '/acg/b3/': {'S1': 'acgb1', 'S2': 'acgb2', 'S3': 'acgb3', 'S4': 'acgb4', 'S5': 'acgb5'},
    '/acg/b4/': {'S1': 'acgb1', 'S2': 'acgb2', 'S3': 'acgb3', 'S4': 'acgb4', 'S5': 'acgb5'},
    '/acg/b5/': {'S1': 'acgb1', 'S2': 'acgb2', 'S3': 'acgb3', 'S4': 'acgb4', 'S5': 'acgb5'},
    '/fc/b1/':  {'S1': 'fcb1',  'S2': 'fcb2',  'S3': 'fcb3',  'S4': 'fcb4',  'S5': 'fcb5'},
    '/fc/b2/':  {'S1': 'fcb1',  'S2': 'fcb2',  'S3': 'fcb3',  'S4': 'fcb4',  'S5': 'fcb5'},
    '/fc/b3/':  {'S1': 'fcb1',  'S2': 'fcb2',  'S3': 'fcb3',  'S4': 'fcb4',  'S5': 'fcb5'},
    '/fc/b4/':  {'S1': 'fcb1',  'S2': 'fcb2',  'S3': 'fcb3',  'S4': 'fcb4',  'S5': 'fcb5'},
    '/fc/b5/':  {'S1': 'fcb1',  'S2': 'fcb2',  'S3': 'fcb3',  'S4': 'fcb4',  'S5': 'fcb5'},
    # '/bai_2020/': {'S1': 'pooled'},
    '/takasugi_2024/aorta/':  {'S1': 'Aorta'},
    '/takasugi_2024/brain/':  {'S2': 'Brain'},
    '/takasugi_2024/heart/':  {'S3': 'Heart'},
    '/takasugi_2024/kidney/': {'S4': 'Kidney'},
    '/takasugi_2024/liver/':  {'S5': 'Liver'},
    '/takasugi_2024/lung/':   {'S6': 'Lung'},
    '/takasugi_2024/muscle/': {'S7': 'Muscle'},
    '/takasugi_2024/skin/':   {'S8': 'Skin'},
}

# ── Sample-type classifier rules (order matters: ADPD before AD) ─────────────
SAMPLE_TYPES = [
    # Ping 2018 — disease groups
    ('ADPD', 'ADPD'),
    ('AD',   'AD'),
    ('PD',   'PD'),
    ('CTRL', 'CTRL'),
    # # Bai 2020 — disease groups
    # ('HPC',  'HPC'),
    # ('LPC',  'LPC'),
    # ('MCI',  'MCI'),
    # ('PSP',  'PSP'),
    # Takasugi 2024 — age timepoints
    ('t06mo', 't06mo'),
    ('t15mo', 't15mo'),
    ('t24mo', 't24mo'),
    ('t30mo', 't30mo'),
]

print("Directories loaded successfully!")
print(f"  Datasets registered: {datasets}")


Loading directories...
Directories loaded successfully!
  Datasets registered: (['acgb1', 'acgb2', 'acgb3', 'acgb4', 'acgb5'], ['fcb1', 'fcb2', 'fcb3', 'fcb4', 'fcb5'], ['Aorta', 'Brain', 'Heart', 'Kidney', 'Liver', 'Muscle', 'Skin'])


## Helpers

In [3]:
# Pickle compatibility shim for legacy numpy/pandas pickles
# Thus all scripts should work regardless of what version of these packages you pickled files with on HPC
class CompatUnpickler(pickle.Unpickler):
    # Mapping of old module paths to new ones for numpy and pandas, to handle pickles created with older versions of these libraries
    REMAP = {
        'numpy.core._multiarray_umath': 'numpy._core._multiarray_umath',
        'numpy.core.multiarray':        'numpy._core.multiarray',
        'numpy.core.numeric':           'numpy._core.numeric',
        'numpy.core':                   'numpy._core',
        'pandas.core.indexes.numeric':  'pandas.core.indexes.base',
        'pandas.core.indexes.int64':    'pandas.core.indexes.base',
        'pandas.core.indexes.float64':  'pandas.core.indexes.base',
    }
    def find_class(self, module, name):
        return super().find_class(self.REMAP.get(module, module), name)

def compat_load(path):
    # Load a pickle file using the CompatUnpickler to handle legacy numpy/pandas module paths
    with open(path, 'rb') as f:
        return CompatUnpickler(f).load()

# Experiment / sample-type labels
EXPERIMENT_PREFIXES   = {'acg': 'Ping_2018/acg', 'fc': 'Ping_2018/fc', 'pooled': 'Bai_2020'}
DATASET_EXP_OVERRIDES = {t: 'Takasugi_2024' for t in ms_tissues}

def get_experiment_label(dataset_name):
    # First check for dataset-specific override, then fall back to prefix matching, then default to 'other'
    if dataset_name in DATASET_EXP_OVERRIDES:
        return DATASET_EXP_OVERRIDES[dataset_name]
    for prefix, experiment in EXPERIMENT_PREFIXES.items():
        if dataset_name.lower().startswith(prefix):
            return experiment
    return 'other'

def classify_sample(sample_name):
    # Classify sample type based on substring matching rules defined in SAMPLE_TYPES
    for pattern, label in SAMPLE_TYPES:
        if pattern in sample_name:
            return label
    return 'Unknown'

def exp_dir(experiment):
    # Get directory path for a given experiment, creating it if it doesn't exist
    path = os.path.join(DEPDIR, experiment)
    os.makedirs(path, exist_ok=True)
    return path

def resolve_dataset_key(proj):
    # Resolve dataset name from project directory path using substring matching rules defined in tmt_to_dataset
    d = proj.lower()
    for key in tmt_to_dataset:
        if key.lower() in d:
            return tmt_to_dataset[key]
    return {}


# Generate Analysis Dependencies

## 1. `dataset_metrics.xlsx`

In [4]:
print("=== Generating: dataset_metrics.xlsx ===")

rows = []
for ds, mq_dir in zip(all_datasets, mq_dir_list):
    # Read MaxQuant output files for the dataset
    msms     = pd.read_csv(mq_dir + 'msms.txt',     sep='\t', low_memory=False)
    peptides = pd.read_csv(mq_dir + 'peptides.txt', sep='\t', low_memory=False)
    evidence = pd.read_csv(mq_dir + 'evidence.txt', sep='\t', low_memory=False)

    # Compile metrics for the dataset and append to rows list
    rows.append([
        ds,
        get_experiment_label(ds),
        msms[msms['PEP'] < 0.01].shape[0],
        peptides['Sequence'].nunique(),
        evidence['Sequence'].nunique(),
        peptides[peptides['Unique (Proteins)'] == 'yes']['Sequence'].nunique(),
    ])
    print(f"  {ds}: {rows[-1][2]:,} PSMs | {rows[-1][3]:,} peptides")

# Save metrics to Excel files, one per experiment
metrics_all = pd.DataFrame(rows, columns=[
    'Dataset', 'Experiment', 'Identified PSMs (1% FDR)',
    'Peptides', 'Peptides (evidence)', 'Unique peptides',
])

# Group by experiment and save each group's metrics to a separate Excel file
for exp, grp in metrics_all.groupby('Experiment'):
    out = os.path.join(exp_dir(exp), 'dataset_metrics.xlsx')
    grp.drop(columns='Experiment').reset_index(drop=True).to_excel(
        out, index=False, engine='openpyxl')
    print(f"  Saved → {out}")


=== Generating: dataset_metrics.xlsx ===
  acgb1: 112,553 PSMs | 71,335 peptides
  acgb2: 103,223 PSMs | 71,720 peptides
  acgb3: 101,486 PSMs | 72,031 peptides
  acgb4: 83,609 PSMs | 60,342 peptides
  acgb5: 91,758 PSMs | 65,912 peptides
  fcb1: 62,697 PSMs | 52,093 peptides
  fcb2: 57,476 PSMs | 46,951 peptides
  fcb3: 55,302 PSMs | 44,672 peptides
  fcb4: 70,859 PSMs | 59,433 peptides
  fcb5: 55,144 PSMs | 42,233 peptides
  Aorta: 58,435 PSMs | 44,404 peptides
  Brain: 76,660 PSMs | 53,704 peptides
  Heart: 40,195 PSMs | 33,982 peptides
  Kidney: 54,111 PSMs | 45,523 peptides
  Liver: 69,138 PSMs | 45,767 peptides
  Muscle: 26,590 PSMs | 18,931 peptides
  Skin: 73,354 PSMs | 56,159 peptides
  Saved → /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/acg/dataset_metrics.xlsx
  Saved → /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/fc/dataset_metrics.xlsx
  Saved → /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Takasugi_2024/dataset_metrics.xls

## 2 + 3. `Position_probability_fragment_ion_data.csv` and `Fragment_ion_dict.p`

For each SAAP/BP pair, looks up the MS/MS spectrum with the most fragment matches
and records the four boundary b/y ions (and their masses) flanking the
substituted residue. Per-experiment positional-probability CSV and a
fragment-mass dictionary are saved together since they are built in the same
pass.


In [5]:
print("=== Generating: Position_probability_fragment_ion_data.csv + Fragment_ion_dict.p ===")

POS_COLS = [
    # Core data
    'SAAP', 'BP', 'AAS', 'AAS index', 'Positional probability', 'charge',
    'TMT set', 'Dataset',
    'saap_b_left_frag',  'saap_b_left_frag_mass',
    'saap_b_right_frag', 'saap_b_right_frag_mass',
    'saap_y_left_frag',  'saap_y_left_frag_mass',
    'saap_y_right_frag', 'saap_y_right_frag_mass',
    'bp_b_left_frag',    'bp_b_left_frag_mass',
    'bp_b_right_frag',   'bp_b_right_frag_mass',
    'bp_y_left_frag',    'bp_y_left_frag_mass',
    'bp_y_right_frag',   'bp_y_right_frag_mass',
]
# Define lists of fragment name and mass columns for easier processing later
FRAG_NAME_COLS = [c for c in POS_COLS if c.endswith('_frag')]
FRAG_MASS_COLS = [c for c in POS_COLS if c.endswith('_frag_mass')]

exp_rows, exp_frag, exp_count = {}, {}, {}

for ds, proj, samples, mq_dir in zip(all_datasets, proj_dir_list, samples_list, mq_dir_list):
    # Resolve dataset name from project directory path
    exp = get_experiment_label(ds)
    exp_rows.setdefault(exp, [])
    exp_frag.setdefault(exp, {})
    exp_count.setdefault(exp, 0)

    mtp_dict = compat_load(proj + 'Ion_validated_MTP_dict.p')
    msms     = pd.read_csv(mq_dir + 'msms.txt', sep='\t', low_memory=False)

    for s in samples:
        # If the TMT set doesn't have mistranslation data, skip it
        if s not in mtp_dict:
            continue
        s_df = pd.DataFrame.from_dict(mtp_dict[s])[
            ['mistranslated sequence', 'DP Base Sequence', 'aa subs',
             'origin aa index', 'aa subs positional probability', 'Charge']
        ].copy()
        s_df['TMT set'] = s
        s_df['Dataset'] = ds
        s_df.reset_index(drop=True, inplace=True)
        for c in FRAG_NAME_COLS: s_df[c] = ''
        for c in FRAG_MASS_COLS: s_df[c] = np.nan

        for i, row in s_df.iterrows():
        # Extract relevant data for the current row
            saap, bp, idx = row['mistranslated sequence'], row['DP Base Sequence'], row['origin aa index']
            global_i = i + exp_count[exp]
            exp_frag[exp][global_i] = {}

            for peptide, mapping, frag_key in [
                # Mapping of fragment names to column prefixes for SAAP and BP sequences, 
                # using the index to determine which fragments to look for
                (saap, [
                    ('b' + str(idx),                  'saap_b_left'),
                    ('b' + str(idx + 1),              'saap_b_right'),
                    ('y' + str(len(saap) - idx + 1),  'saap_y_left'),
                    ('y' + str(len(saap) - idx),      'saap_y_right'),
                ], 'SAAP'),
                (bp, [
                    ('b' + str(idx),                  'bp_b_left'),
                    ('b' + str(idx + 1),              'bp_b_right'),
                    ('y' + str(len(bp) - idx + 1),    'bp_y_left'),
                    ('y' + str(len(bp) - idx),        'bp_y_right'),
                ], 'BP'),
            ]:
                # Find all MS/MS hits for the peptide sequence, 
                # and extract fragment ion information for the best hit (lowest PEP)
                hits = msms[msms['Sequence'] == peptide].sort_values(
                    'Number of matches', ascending=False)
                if hits.empty:
                    continue
                best   = hits.iloc[0]
                names  = best['Matches'].split(';')
                masses = best['Masses'].split(';')
                frags  = {n: m for n, m in zip(names, masses)
                          if ('b' in n or 'y' in n) and '-' not in n}
                exp_frag[exp][global_i][frag_key] = frags
                for frag_name, col_prefix in mapping:
                    s_df.loc[i, col_prefix + '_frag'] = frag_name
                    if frag_name in frags:
                        s_df.loc[i, col_prefix + '_frag_mass'] = float(frags[frag_name])
         
        # Append the processed DataFrame for the current sample to the list of rows for the experiment,
        # and update the count of total rows for the experiment to ensure unique global indices for fragment ion data
        exp_rows[exp].append(s_df)
        exp_count[exp] += len(s_df)

for exp, dfs in exp_rows.items():
    # Concatenate all sample DataFrames for the experiment into a single DataFrame, 
    # save it to CSV, and save the corresponding fragment ion dictionary to a pickle file
    if not dfs:
        continue
    pos_df = pd.concat(dfs)
    pos_df.columns = POS_COLS

    edir     = exp_dir(exp)
    csv_path = os.path.join(edir, 'Position_probability_fragment_ion_data.csv')
    pkl_path = os.path.join(edir, 'Fragment_ion_dict.p')

    pos_df.to_csv(csv_path)
    with open(pkl_path, 'wb') as f:
        pickle.dump(exp_frag[exp], f)
    print(f"  Saved to {csv_path}")
    print(f"  Saved to {pkl_path}")

=== Generating: Position_probability_fragment_ion_data.csv + Fragment_ion_dict.p ===
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/acg/Position_probability_fragment_ion_data.csv
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/acg/Fragment_ion_dict.p
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/fc/Position_probability_fragment_ion_data.csv
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/fc/Fragment_ion_dict.p
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Takasugi_2024/Position_probability_fragment_ion_data.csv
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Takasugi_2024/Fragment_ion_dict.p


## 4. `SAAP_precursor_reporter_quant.xlsx`

Per-sample SAAP quantification table. Joins precursor + reporter intensities
from `MTP_quant_dict.p`, the minimum PEP for each peptide from MaxQuant
`evidence.txt`, and positional probability from the file produced in the
previous step.


In [6]:
print("=== Generating: SAAP_precursor_reporter_quant.xlsx ===")

# Load just-saved positional probabilities across all experiments
pos_prob_all = pd.concat(
    [pd.read_csv(os.path.join(DEPDIR, exp, 'Position_probability_fragment_ion_data.csv'),
                 index_col=0)
     for exp in {get_experiment_label(d) for d in all_datasets}
     if os.path.exists(os.path.join(DEPDIR, exp,
                                    'Position_probability_fragment_ion_data.csv'))],
    ignore_index=True,
)

rows = []
for ds, proj, mq_dir in zip(all_datasets, proj_dir_list, mq_dir_list):
    # Load MaxQuant evidence file to get PEP values for SAAP and BP peptides, 
    # and load the MTP quantification dictionary for the dataset
    MTP_quant_dict = compat_load(proj + 'MTP_quant_dict.p')

    ev = pd.read_csv(mq_dir + 'evidence.txt', sep='\t', low_memory=False)
    ev['_pep'] = ev['Sequence'].astype(str).str.strip().str.upper()
    pep_lookup = ev.groupby('_pep')['PEP'].min().to_dict()

    ds_map = resolve_dataset_key(proj)

    for entry in MTP_quant_dict.values():
        # Extract relevant data for the current entry, including sequences, 
        # amino acid substitution, TMT sets, and lookup PEP values for SAAP and BP peptides
        mtp_seq  = str(entry['MTP_seq']).strip().upper()
        bp_seq   = str(entry['BP_seq']).strip().upper()
        aa_sub   = entry['aa_sub']
        tmt_sets = entry['tmt_sets']
        saap_pep = pep_lookup.get(mtp_seq, np.nan)
        bp_pep   = pep_lookup.get(bp_seq,  np.nan)

        for sample, sd in entry['Patient_dict'].items():
            # For each TMT set associated with the entry, look up the positional probability 
            # from the combined DataFrame of all experiments,
            # and compile all relevant data into a row for the final report
            for tmt_set in tmt_sets:
                dataset = ds_map.get(tmt_set, 'Unknown')
                m = pos_prob_all[
                    (pos_prob_all['SAAP']    == mtp_seq) &
                    (pos_prob_all['BP']      == bp_seq)  &
                    (pos_prob_all['TMT set'] == tmt_set) &
                    (pos_prob_all['Dataset'] == dataset)
                ]
                pos_prob = m.iloc[0]['Positional probability'] if not m.empty else np.nan

                rows.append({
                    'Dataset':                dataset,
                    'Experiment':             get_experiment_label(dataset),
                    'MTP_seq':                mtp_seq,
                    'BP_seq':                 bp_seq,
                    'aa_sub':                 aa_sub,
                    'tmt_set':                tmt_set,
                    'Positional_probability': pos_prob,
                    'SAAP_PEP':               saap_pep,
                    'BP_PEP':                 bp_pep,
                    'MTP_PrecInt':            entry['MTP_PrecInt'].get(tmt_set),
                    'BP_PrecInt':             entry['BP_PrecInt'].get(tmt_set),
                    'Prec_RAAS':              entry['Prec_ratio'].get(tmt_set),
                    'Sample':                 sample,
                    'Sample Type':            classify_sample(sample),
                    'MTP_ReportInt':          sd['MTP_ReportInt'],
                    'BP_ReportInt':           sd['BP_ReportInt'],
                    'Reporter_RAAS':          sd['Reporter_ratio'],
                    'BP_ReportInt_Norm':      sd['BP_ReportInt_Norm'],
                    'MTP_ReportInt_Norm':     sd['MTP_ReportInt_Norm'],
                })

saap_df = pd.DataFrame(rows)
saap_df.insert(0, 'Row Number', range(len(saap_df)))

for exp, grp in saap_df.groupby('Experiment'):
    # For each experiment, drop the 'Experiment' column, reset the index, 
    # add a 'Row Number' column, and save to an Excel file
    grp = grp.drop(columns='Experiment').reset_index(drop=True)
    grp['Row Number'] = range(len(grp))
    out = os.path.join(exp_dir(exp), 'SAAP_precursor_reporter_quant.xlsx')
    grp.to_excel(out, index=False)
    print(f"  Saved to {out}  ({len(grp):,} rows)")


=== Generating: SAAP_precursor_reporter_quant.xlsx ===
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/acg/SAAP_precursor_reporter_quant.xlsx  (5,706 rows)
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/fc/SAAP_precursor_reporter_quant.xlsx  (2,376 rows)
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Takasugi_2024/SAAP_precursor_reporter_quant.xlsx  (16,576 rows)


## 5. `MTP_sequences.fasta`

In [7]:
print("=== Generating: MTP_sequences.fasta ===")

exp_lines = {}
for ds, proj in zip(all_datasets, proj_dir_list):
    # Load the MTP quantification dictionary for the dataset and extract MTP sequences and amino acid substitutions,
    # organizing them by experiment for FASTA output
    MTP_dict = compat_load(proj + 'MTP_quant_dict.p')
    exp = get_experiment_label(ds)
    exp_lines.setdefault(exp, [])
    for idx, entry in MTP_dict.items():
        mtp_seq = str(entry['MTP_seq']).strip().upper()
        aa_sub  = str(entry['aa_sub']).replace(',', '_').replace(' ', '')
        exp_lines[exp].extend([f">{ds}_MTP_{idx}_AAsub_{aa_sub}", mtp_seq])

for exp, lines in exp_lines.items():
    # For each experiment, save the compiled MTP sequences and their headers to a FASTA file
    out = os.path.join(exp_dir(exp), 'MTP_sequences.fasta')
    with open(out, 'w') as f:
        f.write('\n'.join(lines))
    print(f"  Saved to {out}  ({len(lines)//2:,} sequences)")

=== Generating: MTP_sequences.fasta ===
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/acg/MTP_sequences.fasta  (634 sequences)
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/fc/MTP_sequences.fasta  (264 sequences)
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Takasugi_2024/MTP_sequences.fasta  (1,036 sequences)


## 6. `Modified_peptide_filter_dict_DP2valSAAP.p`

Funnel of peptide counts from raw evidence → DP → PTM → AAS → high-confidence
→ ion-validated. Used to render the filter waterfall plot.


In [8]:
print("=== Generating: Modified_peptide_filter_dict_DP2valSAAP.p ===")

FILTER_FILES = [
    # List of files to load for each dataset, along with the key to extract 
    # from the loaded dictionary and a description of the type of samples they contain
    ('DP_search_evidence_dict.p', 'main', 'Raw file'),
    ('DP_dict.p',                 'dp',   'Raw file'),
    ('PTM_dict.p',                'ptm',  'Raw file'),
    ('MTP_dict.p',                'mtp',  'Raw file'),
    ('qMTP_dict.p',               'qmtp', 'mistranslated sequence'),
    ('Ion_validated_MTP_dict.p',  'ion',  'Raw file'),
]

exp_data = {}
for ds, proj, samples in zip(all_datasets, proj_dir_list, samples_list):
    # Load the specified filter files for the dataset, 
    # extracting the relevant dictionaries based on the provided keys, 
    # and compile counts of peptides passing each filter for each sample
    parts = {key: compat_load(proj + fname) for fname, key, _ in FILTER_FILES}

    rows = [[s] + [len(parts[key][s][col]) for _, key, col in FILTER_FILES]
            for s in samples]
    df = pd.DataFrame(rows, columns=[
        'TMT set', 'Main peptides', 'DP', 'PTM', 'AAS',
        'High-confidence', 'Validated',
    ])
    df['Dataset'] = ds
    exp_data.setdefault(get_experiment_label(ds), {})[ds] = df

for exp, ds_dfs in exp_data.items():
    # For each experiment, save the compiled filter data for all datasets to a pickle file
    out = os.path.join(exp_dir(exp), 'Modified_peptide_filter_dict_DP2valSAAP.p')
    with open(out, 'wb') as f:
        pickle.dump(ds_dfs, f)
    print(f"  Saved → {out}")


=== Generating: Modified_peptide_filter_dict_DP2valSAAP.p ===
  Saved → /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/acg/Modified_peptide_filter_dict_DP2valSAAP.p
  Saved → /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/fc/Modified_peptide_filter_dict_DP2valSAAP.p
  Saved → /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Takasugi_2024/Modified_peptide_filter_dict_DP2valSAAP.p


## 7 + 8. `PTM_heatmap_dict.p` and `PTM_heatmap_data.xlsx`

Per-dataset PTM count matrix (rows = PTMs, cols = TMT plex), then the shared
top-N PTMs across all datasets in an experiment, normalized by peptides per
1000 (`Peptides (evidence)` from `dataset_metrics.xlsx`).


In [9]:
print("=== Generating: PTM_heatmap_dict.p + PTM_heatmap_data.xlsx ===")

TOP_N     = 40
N_SAMPLES = 23

def build_ptm_count_df(ptm_dict):
    # Build a DataFrame counting the occurrences of each PTM across samples for a given PTM dictionary,
    # ensuring that all PTMs are included as rows and all samples as columns, with missing values filled with zeros
    master = []
    for v in ptm_dict.values():
        for ptms in v['PTM'].values():
            for p in ptms:
                if p not in master:
                    master.append(p)
    df = pd.DataFrame(index=master, columns=range(1, N_SAMPLES + 1))
    for s, v in ptm_dict.items():
        try:    s_int = int(s[1:])
        except: s_int = s
        cnt = Counter(p for sub in v['PTM'].values() for p in sub)
        for p, c in cnt.items():
            df.loc[p, s_int] = c
    return df.fillna(0).astype(float)

exp_ptm = {}
for ds, proj in zip(all_datasets, proj_dir_list):
    # Load the PTM dictionary for the dataset, build a DataFrame counting PTM occurrences across samples,
    # and store it in a dictionary organized by experiment and dataset for later saving and plotting
    exp = get_experiment_label(ds)
    df  = build_ptm_count_df(compat_load(proj + 'PTM_dict.p'))
    df['avg'] = df.mean(axis=1)
    df.sort_values('avg', ascending=False, inplace=True)
    df.index = [x[0].upper() + x[1:] for x in df.index]
    exp_ptm.setdefault(exp, {})[ds] = df

for exp, ptm_heatmap_dict in exp_ptm.items():
    # For each experiment, save the compiled PTM heatmap data for all datasets to a pickle file,
    # and also compile a DataFrame of the top PTMs across datasets, normalized by the 
    # number of peptides in each dataset, and save it to an Excel file for plotting
    edir = exp_dir(exp)

    pkl_path = os.path.join(edir, 'PTM_heatmap_dict.p')
    with open(pkl_path, 'wb') as f:
        pickle.dump(ptm_heatmap_dict, f)
    print(f"  Saved → {pkl_path}")

    metrics = pd.read_excel(os.path.join(edir, 'dataset_metrics.xlsx'))

    top = None
    for df in ptm_heatmap_dict.values():
        # Identify the top N PTMs in each dataset and find the intersection of 
        # these top PTMs across all datasets in the experiment,
        # to focus the heatmap on the most commonly observed PTMs
        s   = set(df.index[:TOP_N])
        top = s if top is None else top & s
    if not top:
        print(f"  WARNING: no shared top-{TOP_N} PTMs in '{exp}' — skipping data file")
        continue

    plot_df = pd.DataFrame({
        # Compile a DataFrame containing the average counts of the top PTMs for each dataset,
        # normalized by the number of peptides in each dataset to account for differences in dataset size
        ds: ptm_heatmap_dict[ds].loc[list(top), 'avg']
        for ds in ptm_heatmap_dict
    }).astype(float)

    for ds in plot_df.columns:
        # Normalize the average PTM counts for each dataset by the number of peptides (in thousands) in that dataset,
        # to allow for fair comparison of PTM prevalence across datasets of different sizes
        n = metrics.loc[metrics['Dataset'] == ds, 'Peptides (evidence)'].values
        if len(n):
            plot_df[ds] = plot_df[ds] / (n[0] / 1000)

    xlsx_path = os.path.join(edir, 'PTM_heatmap_data.xlsx')
    plot_df.to_excel(xlsx_path)
    print(f"  Saved to {xlsx_path}")


=== Generating: PTM_heatmap_dict.p + PTM_heatmap_data.xlsx ===


/var/folders/g1/2zxfph5533g5zjz997szslmw0000gn/T/ipykernel_31749/1587355368.py:22: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.fillna(0).astype(float)
/var/folders/g1/2zxfph5533g5zjz997szslmw0000gn/T/ipykernel_31749/1587355368.py:22: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.fillna(0).astype(float)
/var/folders/g1/2zxfph5533g5zjz997szslmw0000gn/T/ipykernel_31749/1587355368.py:22: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(

  Saved → /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/acg/PTM_heatmap_dict.p
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/acg/PTM_heatmap_data.xlsx
  Saved → /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/fc/PTM_heatmap_dict.p
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/fc/PTM_heatmap_data.xlsx
  Saved → /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Takasugi_2024/PTM_heatmap_dict.p
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Takasugi_2024/PTM_heatmap_data.xlsx


## 9. `fragments_per_saap_4barplot_allDS.xlsx`

For each validated SAAP, counts fragment ions (b/y) crossing the substituted
residue across all validation-search evidence scans. Bins per experiment into
{0, 1, 2-5, 6-10, >10}; bins 0 and 1 are flagged "Used = No".

Note: this step also writes `fragment_evidence` back into each
`Validated_MTP_dict.p` for downstream use.


In [10]:
print("=== Generating: fragments_per_saap_4barplot_allDS.xlsx ===")

# Define categories for the number of fragment ions providing evidence for the mistranslated amino acid position,
# which will be used for binning the data in the final bar plot
CATS = ['0', '1', '2-5', '6-10', '>10']

def n_frags_over_mtp(matches, mtp, sub_idx):
    # Count the number of b/y fragment ions in the MS/MS matches that provide evidence 
    # for the mistranslated amino acid position,
    # while excluding common neutral losses and modifications, 
    # and ensuring that the fragments are relevant to the position of interest in the peptide sequence
    count = 0
    for f in matches:
        if any(t in f for t in ('NH3', 'H2O', '(', 'a')):
            continue
        if 'b' in f and int(f[1:]) > sub_idx:
            count += 1
        elif 'y' in f and len(mtp) - int(f[1:]) <= sub_idx:
            count += 1
    return count

exp_counts = {}
for ds, proj, samples, mq_dir in zip(all_datasets, proj_dir_list, samples_list, mq_dir_list):
    # Load the MTP dictionary and MS/MS data for the dataset, and for each sample, 
    # count the number of fragment ions providing evidence for the mistranslated amino acid position in each MTP entry,
    # categorizing the counts into defined bins and compiling them for each experiment for later saving and plotting
    exp = get_experiment_label(ds)
    exp_counts.setdefault(exp, [0, 0, 0, 0, 0])

    val_path  = proj + 'Validated_MTP_dict.p'
    mtp_dict  = compat_load(val_path)
    ev_dict   = compat_load(proj + 'Validation_search_evidence_dict.p')
    msms      = pd.read_csv(mq_dir + 'msms.txt', sep='\t', low_memory=False)

    # Map (Raw file, Scan number) -> Matches string. Column order in msms.txt is NOT
    # (Raw file, Scan number, ...): index 1 is 'Fraction', and 'Scan number' lives further
    # along, so a (raw, scan) lookup must go by column name, not position.
    msms_lookup = {
        (r, sc): m
        for r, sc, m in zip(msms['Raw file'], msms['Scan number'], msms['Matches'])
    }

    print(f"  Processing {ds}")

    for s in samples:
        # If the TMT set doesn't have validated MTP data, skip it
        if s not in mtp_dict:
            continue
        ev = ev_dict[s]
        mtp_dict[s]['fragment_evidence'] = {}

        for k in mtp_dict[s]['aa subs']:
            seq     = mtp_dict[s]['mistranslated sequence'][k]
            bp      = mtp_dict[s]['DP Base Sequence'][k]
            sub_idx = next(i for i, x in enumerate(bp) if seq[i] != x)
            ev_idx  = mtp_dict[s]['idx_val_evidence'][k]
            best    = 0

            for idx in ev_idx:
                row  = ev.iloc[idx]
                raw  = row['Raw file']
                scan = row['MS/MS scan number']
                matches_entry = msms_lookup.get((raw, scan))
                if isinstance(matches_entry, str) and matches_entry.strip():
                    n = n_frags_over_mtp(matches_entry.split(';'), seq, sub_idx)
                    if n > best:
                        best = n
            mtp_dict[s]['fragment_evidence'][k] = best

    with open(val_path, 'wb') as f:
        pickle.dump(mtp_dict, f)

    for s in samples:
        # For each sample, look up the number of fragment ions providing evidence for the mistranslated amino acid position in the MTP dictionary,
        # categorize the counts into defined bins, and update the counts for the experiment accordingly
        if s not in mtp_dict:
            continue
        for n in mtp_dict[s].get('fragment_evidence', {}).values():
            if   n == 0:  exp_counts[exp][0] += 1
            elif n == 1:  exp_counts[exp][1] += 1
            elif n <= 5:  exp_counts[exp][2] += 1
            elif n <= 10: exp_counts[exp][3] += 1
            else:         exp_counts[exp][4] += 1

for exp, counts in exp_counts.items():
    out = os.path.join(exp_dir(exp), 'fragments_per_saap_4barplot_allDS.xlsx')
    df  = pd.DataFrame(zip(CATS, counts), columns=['Bin', 'Count'])
    df['Used'] = ['No' if b in ('0', '1') else 'Yes' for b in df['Bin']]
    df.to_excel(out)
    print(f"  Saved to {out}")


=== Generating: fragments_per_saap_4barplot_allDS.xlsx ===
  Processing acgb1
  Processing acgb2
  Processing acgb3
  Processing acgb4
  Processing acgb5
  Processing fcb1
  Processing fcb2
  Processing fcb3
  Processing fcb4
  Processing fcb5
  Processing Aorta
  Processing Brain
  Processing Heart
  Processing Kidney
  Processing Liver
  Processing Muscle
  Processing Skin
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/acg/fragments_per_saap_4barplot_allDS.xlsx
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/fc/fragments_per_saap_4barplot_allDS.xlsx
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Takasugi_2024/fragments_per_saap_4barplot_allDS.xlsx


In [11]:
print("=== Generating genome_substr_dict.p files ===")

GENOME_SUBSTR_KEYS = [
    '1-frame genome substring',
    '2-frame genome substring',
    '3-frame genome substring',
    '4-frame genome substring',
    '5-frame genome substring',
    '6-frame genome substring',
]

for proj_dir in proj_dir_list:
    dp_path = os.path.join(proj_dir, 'DP_dict.p')
    if not os.path.exists(dp_path):
        print(f"Skipping missing DP_dict.p: {dp_path}")
        continue

    print(f"Processing: {dp_path}")
    try:
        dp_dict = pickle.load(open(dp_path, 'rb'))
        genome_substr_dict = {}
        all_mtps = set()
        frame_hits = {
            1: set(),
            2: set(),
            3: set(),
            4: set(),
            5: set(),
            6: set(),
        }

        # dp_dict structure:
        # dp_dict[sample][field][index]
        for sample in dp_dict.keys():
            mt_seq_dict = dp_dict[sample]['mistranslated sequence']
            for idx, seq_list in mt_seq_dict.items():
                all_mtps.update(seq_list)
                for frame in range(1, 7):
                    key = f'{frame}-frame genome substring'
                    if key not in dp_dict[sample]:
                        continue
                    hit_list = dp_dict[sample][key].get(idx, [])
                    for seq_i, seq in enumerate(seq_list):
                        if (
                            seq_i < len(hit_list)
                            and hit_list[seq_i] == True
                        ):
                            frame_hits[frame].add(seq)

        genome_substr_dict = {
            'MTPs': list(all_mtps),

            'all_frame1_seqs': list(frame_hits[1]),
            'N_all_frame1': len(frame_hits[1]),

            'all_frame2_seqs': list(frame_hits[2]),
            'N_all_frame2': len(frame_hits[2]),

            'all_frame3_seqs': list(frame_hits[3]),
            'N_all_frame3': len(frame_hits[3]),

            'all_frame4_seqs': list(frame_hits[4]),
            'N_all_frame4': len(frame_hits[4]),

            'all_frame5_seqs': list(frame_hits[5]),
            'N_all_frame5': len(frame_hits[5]),

            'all_frame6_seqs': list(frame_hits[6]),
            'N_all_frame6': len(frame_hits[6]),

            # backwards compatibility with older decode notebooks
            'all_1frame_seqs': list(frame_hits[1]),
            'N_all_1frame': len(frame_hits[1]),

            'all_3frame_seqs': list(frame_hits[3]),
            'N_all_3frame': len(frame_hits[3]),

            'all_6frame_seqs': list(frame_hits[6]),
            'N_all_6frame': len(frame_hits[6]),
        }

        out_path = os.path.join(proj_dir, 'genome_substr_dict.p')
        pickle.dump(
            genome_substr_dict,
            open(out_path, 'wb')
        )
        print(f"Saved: {out_path}")

    except Exception as e:
        print(f"ERROR processing {proj_dir}")
        print(str(e))

print("Finished generating genome_substr_dict.p files.")

=== Generating genome_substr_dict.p files ===
Processing: /Users/alexmaropakis/Projects/Project_BrainDecode/Analysis_Inputs/Ping_2018/acg/b1/DP_dict.p
Saved: /Users/alexmaropakis/Projects/Project_BrainDecode/Analysis_Inputs/Ping_2018/acg/b1/genome_substr_dict.p
Processing: /Users/alexmaropakis/Projects/Project_BrainDecode/Analysis_Inputs/Ping_2018/acg/b2/DP_dict.p
Saved: /Users/alexmaropakis/Projects/Project_BrainDecode/Analysis_Inputs/Ping_2018/acg/b2/genome_substr_dict.p
Processing: /Users/alexmaropakis/Projects/Project_BrainDecode/Analysis_Inputs/Ping_2018/acg/b3/DP_dict.p
Saved: /Users/alexmaropakis/Projects/Project_BrainDecode/Analysis_Inputs/Ping_2018/acg/b3/genome_substr_dict.p
Processing: /Users/alexmaropakis/Projects/Project_BrainDecode/Analysis_Inputs/Ping_2018/acg/b4/DP_dict.p
Saved: /Users/alexmaropakis/Projects/Project_BrainDecode/Analysis_Inputs/Ping_2018/acg/b4/genome_substr_dict.p
Processing: /Users/alexmaropakis/Projects/Project_BrainDecode/Analysis_Inputs/Ping_2018/ac

In [12]:
print("All dependencies generated.")

All dependencies generated.
